### LoRA (r8) Model Adapter training:
We'll load and tranform the dataset first and then will train the adapter

In [13]:
TRAINING_DATA_PATH = r"C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\data\splits\training.jsonl"
VALIDATION_DATA_PATH = r"C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\data\splits\validate.jsonl"
TEST_DATA_PATH = r"C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\data\splits\test.jsonl"

In [14]:
# format data in coversational form
def data_format(dataset) -> list:
    result = []
    for example in dataset:
        
        formatted = {
            'messages':[
                {
                    'role':'user',
                    'content':None
                },
                {
                    'role':'assistant',
                    'content':None
                }
            ],
            'intent':None,
            'priority':None
        }

        formatted['messages'][0]['content'] = example.get('input')
        formatted['messages'][1]['content'] = f"INTENT:{example.get('input')} \nPRIORITY:{example.get('priority')} \nACTION:{example.get('action')}"
        formatted['intent']=example.get('intent')
        formatted['priority']=example.get('priority')

        result.append(formatted)
    return result

In [15]:
import json

with open(TRAINING_DATA_PATH, 'r') as f:
    training_data_uf = list(map(json.loads, f.readlines()))

with open(VALIDATION_DATA_PATH, 'r') as f:
    validation_data_uf = list(map(json.loads, f.readlines()))

with open(TEST_DATA_PATH, 'r') as f:
    test_data_uf = list(map(json.loads, f.readlines()))

print("Traning dataset length: ",len(training_data_uf))
print("Validation dataset length: ",len(validation_data_uf))
print('Test dataset length: ',len(test_data_uf))

Traning dataset length:  402
Validation dataset length:  86
Test dataset length:  87


In [16]:
formatted_training_data = data_format(training_data_uf)
formatted_validation_data = data_format(validation_data_uf)
formatted_test_data = data_format(test_data_uf)

In [17]:
print(len(formatted_training_data))
print(len(formatted_validation_data))
print(len(formatted_test_data))

402
86
87


In [18]:
# for model training we'll use the messages only not the metadata
train_dataset = [dict([next(iter(d.items()))]) for d in formatted_training_data]
validation_dataset = [dict([next(iter(d.items()))]) for d in formatted_validation_data]
test_dataset = [dict([next(iter(d.items()))]) for d in formatted_test_data]

In [21]:
TRAIN_SAVE_PATH = r'C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\data\preprocessed\training_data.jsonl'
VALIDATION_SAVE_PATH = r'C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\data\preprocessed\validation_data.jsonl'
TEST_SAVE_PATH = r'C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\data\preprocessed\test_data.jsonl'
with open(TRAIN_SAVE_PATH, 'w') as f:
    for example in train_dataset:
        f.write(json.dumps(example, ensure_ascii=True) + '\n')
with open(VALIDATION_SAVE_PATH, 'w') as f:
    for example in validation_dataset:
        f.write(json.dumps(example, ensure_ascii=True) + '\n')
with open(TEST_SAVE_PATH, 'w') as f:
    for example in test_dataset:
        f.write(json.dumps(example, ensure_ascii=True) + '\n')
    

In [32]:
from datasets import load_dataset
dataset = load_dataset(
    'json',
    data_files={
        'train':TRAIN_SAVE_PATH,
        'validation':VALIDATION_SAVE_PATH,
        'test':TEST_SAVE_PATH
    }
)

Generating train split: 402 examples [00:00, 7923.38 examples/s]
Generating validation split: 86 examples [00:00, 3169.88 examples/s]
Generating test split: 87 examples [00:00, 3269.90 examples/s]


In [34]:
print(dataset,'\n')
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 402
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 86
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 87
    })
}) 

{'messages': [{'role': 'user', 'content': 'My subscription price unexpectedly increased without notification.'}, {'role': 'assistant', 'content': 'INTENT:My subscription price unexpectedly increased without notification. \nPRIORITY:low \nACTION:Explain plan price adjustment and review billing details'}]}


In [8]:
# SFT Config
from trl import SFTConfig, SFTTrainer
training_args = SFTConfig(
    output_dir='exp_02_lora_r8',
    num_train_epochs=3,

    per_device_train_batch_size=3,
    per_device_eval_batch_size=3,

    gradient_accumulation_steps=1,

    learning_rate=2e-4,

    logging_steps=10,

    eval_strategy='epoch',
    save_strategy='epoch',
    use_cpu=True,

    seed=42,
    report_to='none'
)

In [36]:
# we'll laod the lora attached model and follow the workflow
# base model -> inpsction (already done) -> attach lora -> verify trainable params -> SFTTrainer

from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 3927.52it/s]


In [28]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',    # -> don't train bias parameters in adapter
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        'q_proj',
        'v_proj'
    ]
)
from peft import get_peft_model
l_model = get_peft_model(
    model,
    lora_config
)
print(l_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 576, padding_idx=2)
        (layers): ModuleList(
          (0-29): 30 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=576, out_features=576, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=576, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=576, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [40]:
trainer = SFTTrainer(
    model=l_model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
  #peft_config=lora_config  -> since our model is already attached to LoRa, we don't need this
)

In [47]:
# before trainer.train()
print(trainer.model.print_trainable_parameters())
print("Training examples: ", len(dataset['train']))
print("Validation examples: ", len(dataset['validation']))
print('---'*40)
print("TRAINING ARGS: \n",training_args)

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414
None
Training examples:  402
Validation examples:  86
------------------------------------------------------------------------------------------------------------------------
TRAINING ARGS: 
 SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=False,
dataloader_prefetch

### Our Experiment has the following reproducible Config

##### EXP_001_LORA_R8
---
Model:
SmolLM2-135M-Instruct

Method:
LoRA

Rank:
8

Alpha:
16

Dropout:
0.05

Target modules:
q_proj
v_proj

Learning rate:
2e-4

Epochs:
3

Seed:
42

Dataset:
version 1

Evaluation:
validation + fixed test set

In [48]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.329304,1.221933,1.274183,33604.000000,0.783575
2,1.202390,1.143761,1.153279,67208.000000,0.794731
3,1.143245,1.128631,1.140821,100812.000000,0.795441


### The Conceptual Loop: 

TRAINING EXAMPLE  -> Tokenizer -> Token IDs

                    ↓

SmolLM2 -> Predicted token distribution -> Compare with target tokens

                   ↓

Loss -> Backpropagation -> LoRA parameters updated


                   ↓

Next batch


### Remember
Base Model -> not updated 

LoRA Adapter -> updated ✅

In [49]:
adapter_path=r'C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\experiments\exp_01_basline\exp_02_lora_r8\adapter'
trainer.save_model(adapter_path)


In [61]:
# baseline probes
baseline_examples = [
    {
        "id": "baseline_001",
        "input": "I forgot my password and can't access my account.",
        "intent": "password_reset",
        "priority": "medium",
    },
    {
        "id": "baseline_002",
        "input": "Someone changed the email address on my account.",
        "intent": "account_compromise",
        "priority": "high",
    },
    {
        "id": "baseline_003",
        "input": "I was charged twice for the same subscription.",
        "intent": "billing_issue",
        "priority": "medium",
    },
    {
        "id": "baseline_004",
        "input": "Please add dark mode to the application.",
        "intent": "feature_request",
        "priority": "low",
    },
    {
        "id": "baseline_005",
        "input": "I want to close my account permanently.",
        "intent": "account_closure",
        "priority": "low",
    },
]


In [64]:
import torch

def  generate_response(model,tokenizer,user_text : str):
    messages = [
        {
            'role':'user',
            'content':user_text,
        }
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    )
    with torch.no_grad():
        outputs = l_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False #reproducibility
        )
    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [65]:
import json
example_copy = baseline_examples.copy()
for example in example_copy:
    response = generate_response(
        model,
        tokenizer,
        example['input']
    )

    print('-'*100)
    print("INPUT: ", example['input'])
    print("INTENT: ", example['intent'])
    print("MODEL: ", response)
    example['response'] = response

----------------------------------------------------------------------------------------------------
INPUT:  I forgot my password and can't access my account.
INTENT:  password_reset
MODEL:  system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
I forgot my password and can't access my account.
assistant
INTENT:I forgot my password and can't access my account. 
PRIORITY:low 
ACTION:Verify password and restore account
----------------------------------------------------------------------------------------------------
INPUT:  Someone changed the email address on my account.
INTENT:  account_compromise
MODEL:  system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
Someone changed the email address on my account.
assistant
INTENT:Someone changed the email address on my account. 
PRIORITY:high 
ACTION:Request account reset and refund refund account details
------------------------------------------------------------------------------------

In [66]:
PREDICTION_SAVE_PATH = r'C:\Users\saaad kabir\Desktop\Fine Tunnig\finetuning\experiments\exp_01_basline\exp_02_lora_r8\predictions.jsonl'
with open(PREDICTION_SAVE_PATH, 'w') as f:
    for item in example_copy:
        f.write(json.dumps(item) + '\n')

### Takeaways:
As we can see, after training the adapter the model is providing formatted examples. That's a big W but not the end, now we'll check the accuracy, calculate macro f1.